#### Libraries, plot settings and utils

In [ ]:
import numpy as np
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import scipy.stats as stats
from statsmodels.tsa.stattools import acf
import pandas as pd
from scipy.integrate import odeint
from scipy.stats import lognorm
from mpl_toolkits.axes_grid1 import make_axes_locatable


# ---------------------------------------------------------------------
# Plot configuration
# ---------------------------------------------------------------------

plt.style.use("tableau-colorblind10")
cmap = plt.get_cmap("Blues")

plt.rcParams["figure.autolayout"] = True
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9

plt.rcParams.update({
    "font.size": 10,
    "font.family": "serif",
    "font.serif": "cmr10",
    "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True,
})


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

folder_data = "PLACEHOLDER_DATA_FOLDER/"
folder_results = "PLACEHOLDER_RESULTS_FOLDER/"


# ---------------------------------------------------------------------
# Model definitions
# ---------------------------------------------------------------------

def phi(x):
    return np.tanh(x)


def dphi(x):
    return 1 - np.tanh(x) ** 2


def dynamics(x, t, W):
    return -x + np.dot(W, phi(x))


# ---------------------------------------------------------------------
# Network generation
# ---------------------------------------------------------------------

def inter_matrix(K, N, g, gamma):
    J = np.zeros([N, N])

    for i in range(N):
        for j in range(i + 1, N):
            z = np.random.normal(size=2)
            J[i, j] = g / np.sqrt(K) * z[0]
            J[j, i] = g / np.sqrt(K) * (
                gamma * z[0] + np.sqrt(1 - gamma**2) * z[1]
            )

    return J


def adjacency_matrix(N, K, degree_sequence):
    A = np.zeros([N, N])

    for i in range(N):
        for j in range(i + 1, N):
            p_ij = degree_sequence[i] * degree_sequence[j] / (K * N)
            if np.random.rand() < p_ij:
                A[i, j] = 1
                A[j, i] = 1

    return A


def sample_lognormal(mu, sigma, size=1):
    return np.random.lognormal(mean=mu, sigma=sigma, size=size)


# ---------------------------------------------------------------------
# Analysis utilities
# ---------------------------------------------------------------------

def HWHM(acf_values):
    half_max = acf_values[0] / 2
    indices = np.where(acf_values < half_max)[0]

    if len(indices) == 0:
        return None

    i = indices[0]

    x0, x1 = i - 1, i
    y0, y1 = acf_values[x0], acf_values[x1]

    hwhm = x0 + (half_max - y0) * (x1 - x0) / (y1 - y0)
    return hwhm


def ac_sim(sim, N_stat, dt, nlags=None):
    N = sim.shape[1]
    nlags = 100 if nlags is None else nlags

    acfs = []
    acts = []

    for i in range(N):
        C = acf(sim[-N_stat:, i], nlags=nlags - 1)
        acfs.append(C)
        acts.append(HWHM(C) * dt)

    return np.array(acfs), np.array(acts), np.arange(0, nlags * dt, dt)


def ac_plot(acfs, acts, lags):
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))

    ax[0].plot(lags, acfs.T, alpha=0.5)
    ax[0].set_xlabel("Lags")
    ax[0].set_ylabel("ACF")
    ax[0].set_title("Autocorrelation function")

    ax[1].hist(acts, bins=20, alpha=0.5, density=True)
    ax[1].set_xlabel("HWHM")
    ax[1].set_ylabel("Counts")
    ax[1].set_title("HWHM distribution")

    plt.show()


def critical_g(sigma):
    return 1 / np.sqrt(np.exp(sigma**2))


# ---------------------------------------------------------------------
# Simulation
# ---------------------------------------------------------------------

def simulate_lognormal(params, hyperparams, ic=None, path_to_save=None):
    N, g, gamma, mu, sigma = params
    N_steps, dt = hyperparams

    t = np.linspace(0, N_steps * dt, N_steps)

    degree_sequence = sample_lognormal(mu, sigma, N)
    K = int(np.mean(degree_sequence))
    params.append(K)

    A = adjacency_matrix(N, K, degree_sequence)
    J = inter_matrix(K, N, g, gamma)
    W = A * J

    if ic is None:
        sim = odeint(dynamics, np.random.random(N) * 2 - 1, t, args=(W,))
    else:
        sim = odeint(dynamics, ic, t, args=(W,))

    if path_to_save is None:
        return (
            np.array(params, dtype=float),
            sim,
            t,
            A,
            J,
            degree_sequence,
        )

    np.savez(
        path_to_save
        + "lognormal_N{}_g{}_gamma{}_mu{}_sigma{}".format(
            N, g, gamma, mu, sigma
        ),
        params=np.array(params, dtype=float),
        sim=sim,
        t=t,
        A=A,
        J=J,
        degree_sequence=degree_sequence,
    )

    return np.array(params, dtype=float)


def simulate_lognormal_optimized(params, hyperparams):
    N, n, g, gamma, mu, sigma = params
    N_steps, dt, N_stat, N_thinning = hyperparams

    t = np.linspace(0, N_steps * dt, N_steps)

    degree_sequence_pl = sample_lognormal(mu, sigma, N)

    K_true = int(np.mean(degree_sequence_pl))
    A = adjacency_matrix(N, K_true, degree_sequence_pl)
    J = inter_matrix(int(K_true), N, g, gamma)
    W = A * J

    sim = odeint(dynamics, np.random.random(N) * 2 - 1, t, args=(W,))

    f = np.mean(np.std(sim[N_stat::N_thinning, :], axis=1))
    v = np.mean(np.std(sim[N_stat::N_thinning, :], axis=0))

    return [N, n, g, gamma, mu, sigma, K_true, f, v]

#### Data generation

In [ ]:
# ---------------------------------------------------------------------
# Synthetic degree distribution
# ---------------------------------------------------------------------

mu = 3
sigma = 1
N = 10_000

degree_sequence = sample_lognormal(mu, sigma, N)
A = adjacency_matrix(N, np.mean(degree_sequence), degree_sequence)
samples = np.sum(A, axis=0)

x = np.linspace(1, 1000, 10_000)
pdf = lognorm.pdf(x, s=sigma, scale=np.exp(mu))

unique, counts = np.unique(samples, return_counts=True)


# ---------------------------------------------------------------------
# Numerical validation of critical g
# ---------------------------------------------------------------------

N = 1000
dt = 0.05
N_steps = 5000
N_stat = 4000
N_thinning = 25

mu = 3
sigma = 1

g_list = np.linspace(0.3, 1, 14)
gamma = 0
N_samples = 20

params_list = []
for g in g_list:
    for n in range(N_samples):
        params_list.append([N, n, g, gamma, mu, sigma])

hyperparams = [N_steps, dt, N_stat, N_thinning]

print("Number of simulations:", len(params_list))

results = Parallel(n_jobs=-1, verbose=11)(
    delayed(simulate_lognormal_optimized)(params, hyperparams)
    for params in params_list
)

df = pd.DataFrame(
    columns=["N", "n", "g", "gamma", "mu", "sigma", "K_true", "f", "v"],
    data=results,
)

df.to_csv(
    folder_data + "/numerical_results_phase_diagram_mu{}_sigma{}.csv".format(mu, sigma),
    index=False,
)

df = pd.read_csv(
    folder_data + "/numerical_results_phase_diagram_mu{}_sigma{}.csv".format(mu, sigma)
)

df_means = (
    df.groupby(["g"])
    .agg({"K_true": "mean", "f": "mean", "v": "mean"})
    .reset_index()
)

df_stds = (
    df.groupby(["g"])
    .agg({"K_true": "std", "f": "std", "v": "std"})
    .reset_index()
)


# ---------------------------------------------------------------------
# Critical line
# ---------------------------------------------------------------------

sigma_list = np.linspace(0.01, 5, 1000)
g_list = critical_g(sigma_list)


# ---------------------------------------------------------------------
# Example trajectories
# ---------------------------------------------------------------------

np.random.seed(5829)

params_A = [1000, 1, 0, 3, 1]
hyperparams = [1000, 0.1]

results_A = simulate_lognormal(params_A, hyperparams, ic=None, path_to_save=None)
trajectory_A = results_A[1]

params_B = [1000, 0.4, 0, 3, 1]
hyperparams = [1000, 0.1]

results_B = simulate_lognormal(params_B, hyperparams, ic=None, path_to_save=None)
trajectory_B = results_B[1]

time = results_A[2]


# ---------------------------------------------------------------------
# Eigenvalue spectrum
# ---------------------------------------------------------------------

mu = 3
sigma = 1
g = 0.8
N = 2000

log_normal_degree = sample_lognormal(3, 1, N)
A_log_normal = adjacency_matrix(
    N, int(np.mean(log_normal_degree)), log_normal_degree
)
J_log_normal = inter_matrix(
    int(np.mean(log_normal_degree)), N, g, 0
)
W_log_normal = A_log_normal * J_log_normal

log_normal_eigval, log_normal_eigvec = np.linalg.eig(W_log_normal)
log_normal_R = g * np.sqrt(np.exp(sigma**2))

print(np.mean(log_normal_degree))

poisson_degree = np.random.poisson(100, size=N)
A_poisson = adjacency_matrix(N, 100, poisson_degree)
J_poisson = inter_matrix(100, N, g, 0)
W_poisson = A_poisson * J_poisson

poisson_eigval, poisson_eigvec = np.linalg.eig(W_poisson)
poisson_R = g * np.sqrt(np.mean(poisson_degree))

W_full = inter_matrix(N, N, g, 0)
full_eigval, full_eigvec = np.linalg.eig(W_full)
full_R = g

#### Plot Fig.2: phase diagram heterogeneity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import NullLocator
import matplotlib.cm as cm


def combined_figure(
    eigval_Jac, eigvec_Jac, degree_vect, R, cmap,
    eigval_full, eigval_poisson, eigval_log_normal,
    colors, labels,
    sigma_list, g_list, trajectory_A, trajectory_B, time,
    df_means, df_stds, critical_g, sigma,
    unique, counts, N, x, pdf,
    full_red="red", nbins=10,
):
    fig = plt.figure(figsize=(18 / 2.54, 7 / 2.54))

    # ------------------------------------------------------------------
    # Axes positioning
    # ------------------------------------------------------------------

    ax_width, ax_height = 0.18, 0.32

    ax_main = fig.add_axes([0.07, 0.15, 0.32, 0.78])

    ax_hist = fig.add_axes([0.49, 0.6, ax_width, ax_height])
    ax_err = fig.add_axes([0.49, 0.15, ax_width, ax_height])

    ax_jacobian = fig.add_axes([0.77, 0.6, ax_width, ax_height])
    ax_density = fig.add_axes([0.77, 0.15, ax_width, ax_height])

    for ax in [ax_jacobian, ax_density, ax_err, ax_hist, ax_main]:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # ------------------------------------------------------------------
    # Plot A: Jacobian spectrum
    # ------------------------------------------------------------------

    theta = np.linspace(0, 2 * np.pi, 500)
    x_c = R * np.cos(theta) - 1
    y_c = R * np.sin(theta)

    eigvec_abs_sq = np.abs(eigvec_Jac) ** 2
    eigvec_weights = eigvec_abs_sq / np.sum(eigvec_abs_sq, axis=0)
    weighted_degrees = np.dot(degree_vect, eigvec_weights)

    scatter = ax_jacobian.scatter(
        eigval_Jac.real - 1,
        eigval_Jac.imag,
        c=weighted_degrees,
        cmap="plasma",
        s=10,
        alpha=0.6,
        edgecolors="face",
    )

    ax_jacobian.set_xlim(-3, 1)
    ax_jacobian.set_ylim(-1.5, 1.5)

    divider = make_axes_locatable(ax_jacobian)
    cax = divider.append_axes("right", size="5%", pad=0.03)
    cbar = fig.colorbar(scatter, cax=cax)
    cbar.ax.set_ylabel("")
    cax.text(
        0.5, 1.05, r"$\mathcal{D}(\lambda)$",
        ha="center", va="bottom",
        transform=cax.transAxes, fontsize=9
    )

    ax_jacobian.axvline(0, color="black", linestyle="--", lw=2)
    ax_jacobian.plot(x_c, y_c, color="black", lw=2)

    ax_jacobian.set_xlabel(r"$\mathrm{Re}(\lambda)$", labelpad=2)
    ax_jacobian.set_ylabel(r"$\mathrm{Im}(\lambda)$", labelpad=2)

    # ------------------------------------------------------------------
    # Plot B: Radial density
    # ------------------------------------------------------------------

    all_eigvals = [eigval_poisson, eigval_log_normal]
    all_keys = ["poisson", "log_normal"]

    max_radius = max(np.max(np.abs(e)) for e in all_eigvals)
    r_bins = np.linspace(0, max_radius, nbins + 1)
    r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
    dr = np.diff(r_bins)

    colors = {"poisson": "#6B8BD4", "log_normal": "#FD5A40"}

    for eigval, key in zip(all_eigvals, all_keys):
        radii = np.abs(eigval)
        counts2, _ = np.histogram(radii, bins=r_bins)

        annulus_area = np.pi * (r_bins[1:]**2 - r_bins[:-1]**2)
        rho_r = counts2 / annulus_area
        rho_r /= np.sum(rho_r * annulus_area)

        ax_density.bar(
            r_centers,
            rho_r,
            width=dr,
            color=colors[key],
            edgecolor="white",
            alpha=0.7,
            label=labels[key],
        )

    ax_density.set_xlabel(r"$r$", labelpad=2)
    ax_density.set_ylabel(r"$\rho(r)$", labelpad=2)
    ax_density.set_yscale("log")
    ax_density.yaxis.set_minor_locator(NullLocator())
    ax_density.set_ylim(1e-2, 10)

    ax_density.legend(
        fontsize=8,
        markerscale=0.5,
        frameon=False,
        bbox_to_anchor=(0.4, 1),
        loc="upper left",
    )

    # ------------------------------------------------------------------
    # Plot C: Phase diagram
    # ------------------------------------------------------------------

    ax_main.plot(sigma_list, g_list, color="black", lw=2)
    ax_main.set_xlim(0, 3.5)
    ax_main.set_ylim(0, 1.01)

    ax_main.set_xlabel(r"$\sigma$", labelpad=2)
    ax_main.set_ylabel(r"$g$", labelpad=2)

    ax_main.set_yticks([0, 0.5, 1])
    ax_main.set_xticks([0, 1, 2, 3])

    Y_MAX = ax_main.get_ylim()[1]
    ax_main.fill_between(
        sigma_list, g_list, y2=Y_MAX,
        color=cm.plasma(0.85), alpha=0.2
    )
    ax_main.fill_between(
        sigma_list, g_list, y2=0,
        color=cm.plasma(0.2), alpha=0.2
    )

    # Insets
    inset_width = 0.35
    inset_height = 0.25

    inset_A = inset_axes(
        ax_main,
        width=f"{inset_width*100}%",
        height=f"{inset_height*100}%",
        loc="upper right",
        borderpad=1,
    )

    colors_A = cm.plasma(np.linspace(0.85, 0.95, trajectory_A.shape[1]))
    for i in range(min(10, trajectory_A.shape[1])):
        inset_A.plot(time, trajectory_A[:, i], color=colors_A[i], lw=1)

    inset_A.set_xticks([])
    inset_A.set_yticks([])
    inset_A.spines["top"].set_visible(False)
    inset_A.spines["right"].set_visible(False)

    inset_B = inset_axes(
        ax_main,
        width=f"{inset_width*100}%",
        height=f"{inset_height*100}%",
        loc="lower left",
        borderpad=1,
    )

    colors_B = cm.plasma(np.linspace(0.2, 0.4, trajectory_B.shape[1]))
    for i in range(min(10, trajectory_B.shape[1])):
        inset_B.plot(time, trajectory_B[:, i], color=colors_B[i], lw=1)

    inset_B.set_xticks([])
    inset_B.set_yticks([])
    inset_B.spines["top"].set_visible(False)
    inset_B.spines["right"].set_visible(False)

    # ------------------------------------------------------------------
    # Plot D: Error bars
    # ------------------------------------------------------------------

    ax_err.errorbar(
        df_means["g"],
        df_means["f"],
        yerr=df_stds["f"] / np.sqrt(20),
        capsize=2,
        alpha=0.7,
        color=cm.plasma(0.6),
        fmt=".-",
        label="f",
    )

    ax_err.axvline(
        x=critical_g(sigma),
        color="black",
        linestyle="--",
        lw=2,
        label=r"$g_c$",
    )

    ax_err.set_xlabel("g", labelpad=2)
    ax_err.set_ylabel("f", labelpad=2)
    ax_err.set_yticks([0, 0.2, 0.4])
    ax_err.set_xlim(0.3, 0.9)
    ax_err.set_ylim(-0.02, 0.45)

    ax_err.legend(
        frameon=False,
        bbox_to_anchor=(-0.05, 1),
        loc="upper left",
    )

    # ------------------------------------------------------------------
    # Plot E: Degree distribution
    # ------------------------------------------------------------------

    ax_hist.scatter(
        unique,
        counts / N,
        alpha=0.5,
        color=cm.plasma(0.4),
        s=20,
    )

    ax_hist.plot(x, pdf, color=cm.plasma(0.2), lw=2)

    ax_hist.set_xscale("log")
    ax_hist.set_xlim(1, 1000)
    ax_hist.set_xticks([1, 10, 100, 1000])
    ax_hist.set_xticklabels(
        [r"$10^0$", r"$10^1$", r"$10^2$", r"$10^3$"]
    )

    ax_hist.set_ylim([0, 0.04])
    ax_hist.set_yticks([0, 0.02, 0.04])

    ax_hist.set_xlabel(r"$k$", labelpad=2)
    ax_hist.set_ylabel(r"P($k$)", labelpad=2)

    # ------------------------------------------------------------------
    # Panel labels
    # ------------------------------------------------------------------

    for ax, label, x_offset, y_offset in zip(
        [ax_jacobian, ax_density, ax_main, ax_err, ax_hist],
        ["d)", "e)", "a)", "c)", "b)"],
        [-0.3, -0.3, -0.15, -0.3, -0.3],
        [1, 1, 1, 1, 1],
    ):
        ax.text(
            x_offset,
            y_offset,
            label,
            transform=ax.transAxes,
            fontsize=10,
            weight="bold",
            va="top",
            ha="right",
        )

    return fig, cax


fig, cax = combined_figure(
    log_normal_eigval, log_normal_eigvec, log_normal_degree, log_normal_R, "viridis",
    full_eigval, poisson_eigval, log_normal_eigval,
    colors={"full": "blue", "poisson": "green", "log_normal": "orange"},
    labels={"full": "Full", "poisson": "Poisson", "log_normal": "Lognormal"},
    sigma_list=sigma_list, g_list=g_list,
    trajectory_A=trajectory_A, trajectory_B=trajectory_B, time=time,
    df_means=df_means, df_stds=df_stds,
    critical_g=critical_g, sigma=sigma,
    unique=unique, counts=counts,
    N=np.sum(counts),
    x=x, pdf=pdf,
)

fig.savefig(
    folder_results + "panel1.pdf",
    dpi=300,
    pad_inches="tight",
)